In [5]:
!pip install -q sentence-transformers faiss-cpu rank-bm25 fastapi uvicorn pydantic scikit-learn
print("All packages installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 77.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 87.3 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incom

In [3]:
import pandas as pd
import os

# List available files
data_dir = '/kaggle/input/datasets/joykimaiyo18/linkedin-data-jobs-dataset'
for f in sorted(os.listdir(data_dir)):
    print(f)

clean_jobs.csv


In [7]:
df = pd.read_csv('/kaggle/input/datasets/joykimaiyo18/linkedin-data-jobs-dataset/clean_jobs.csv')

In [8]:
print(f"Shape: {df.shape}")
print(f"\nColumns ({len(df.columns)}):")
for col in df.columns:
    non_null = df[col].notna().sum()
    print(f"  {col:30s} ({non_null} non-null, dtype: {df[col].dtype})")
print(f"\n\nFirst 2 rows:")
df.head(2)

Shape: (1048, 10)

Columns (10):
  id                             (1048 non-null, dtype: int64)
  title                          (1048 non-null, dtype: object)
  company                        (1046 non-null, dtype: object)
  location                       (1033 non-null, dtype: object)
  link                           (1048 non-null, dtype: object)
  source                         (959 non-null, dtype: object)
  date_posted                    (1025 non-null, dtype: object)
  work_type                      (0 non-null, dtype: float64)
  employment_type                (0 non-null, dtype: float64)
  description                    (1044 non-null, dtype: object)


First 2 rows:


,id,title,company,location,link,source,date_posted,work_type,employment_type,description
0,1,Data Analyst,Meta,"New York, NY",https://www.linkedin.com/jobs/view/data-analys...,LinkedIn,2025-04-14,NaN,NaN,The Social Measurement team is a growing team ...
1,2,Data Analyst,Meta,"San Francisco, CA",https://www.linkedin.com/jobs/view/data-analys...,LinkedIn,2025-04-14,NaN,NaN,The Social Measurement team is a growing team ...


In [9]:
import pandas as pd
import numpy as np

title_col = next((c for c in ['title', 'job_title', 'Job_Ttl'] if c in df.columns), None)
desc_col = next((c for c in ['description', 'job_description', 'Job_Desc'] if c in df.columns), None)
company_col = next((c for c in ['company_name', 'company', 'Co_Nm'] if c in df.columns), None)
print(f"Title column:       '{title_col}'")
print(f"Description column: '{desc_col}'")
print(f"Company column:     '{company_col}'")
print(f"Total rows:         {len(df)}")

df = df.dropna(subset=[title_col, desc_col]).reset_index(drop=True)
print(f"After cleaning:     {len(df)} rows")

Title column:       'title'
Description column: 'description'
Company column:     'company'
Total rows:         1048
After cleaning:     1044 rows


In [11]:
df.to_csv('/kaggle/working/jobs_processed.csv', index=False)
print(f"\nSaved {len(df)} jobs to jobs_processed.csv")

print(f"\nSample job:")
print(f"  Title:       {df.iloc[0][title_col]}")
if company_col:
    print(f"  Company:     {df.iloc[0][company_col]}")
print(f"  Description: {str(df.iloc[0][desc_col])[:200]}...")


Saved 1044 jobs to jobs_processed.csv

Sample job:
  Title:       Data Analyst
  Company:     Meta
  Description: The Social Measurement team is a growing team with high-visibility within the Communications organization that is being tasked with measuring the efficacy and impact of our social-first Communications...


### Generating the synthetic candidates

In [12]:
import json
import random

random.seed(42)

SKILL_POOLS = {
    "software_engineering": [
        "Python", "Java", "JavaScript", "TypeScript", "C++", "Go", "Rust",
        "Git", "Docker", "Kubernetes", "CI/CD", "Linux", "REST APIs",
        "GraphQL", "Microservices", "System Design", "Agile", "TDD", "OOP",
    ],
    "web_frontend": [
        "React", "Angular", "Vue.js", "Next.js", "HTML5", "CSS3",
        "Tailwind CSS", "Sass", "Webpack", "Redux", "Responsive Design",
        "Figma", "UI/UX Design", "Web Performance",
    ],
    "web_backend": [
        "Node.js", "Express.js", "Django", "Flask", "FastAPI", "Spring Boot",
        "PostgreSQL", "MySQL", "MongoDB", "Redis", "Elasticsearch",
        "RabbitMQ", "Kafka", "OAuth2", "API Design", "Database Design",
    ],
    "data_science": [
        "Python", "R", "SQL", "Pandas", "NumPy", "Scikit-learn",
        "TensorFlow", "PyTorch", "Data Visualization", "Matplotlib",
        "Statistical Analysis", "A/B Testing", "Feature Engineering",
        "Jupyter Notebooks", "Data Wrangling",
    ],
    "machine_learning": [
        "Python", "TensorFlow", "PyTorch", "Scikit-learn", "Hugging Face",
        "NLP", "Computer Vision", "Deep Learning", "MLOps", "MLflow",
        "Transformers", "Recommendation Systems", "Model Deployment",
        "Feature Engineering", "Hyperparameter Tuning",
    ],
    "data_engineering": [
        "Python", "SQL", "Apache Spark", "Apache Airflow", "dbt",
        "Snowflake", "BigQuery", "Databricks", "ETL Pipelines",
        "Data Warehousing", "Kafka", "Terraform", "Data Quality",
    ],
    "devops_cloud": [
        "AWS", "Azure", "GCP", "Docker", "Kubernetes", "Terraform",
        "Ansible", "Jenkins", "GitHub Actions", "Prometheus", "Grafana",
        "Linux", "Bash", "Infrastructure as Code", "SRE",
    ],
    "cybersecurity": [
        "Network Security", "Penetration Testing", "SIEM", "Incident Response",
        "Encryption", "IAM", "OWASP", "Python", "Wireshark", "Nmap",
        "Threat Modeling", "Cloud Security",
    ],
    "mobile_development": [
        "React Native", "Flutter", "Swift", "Kotlin", "iOS Development",
        "Android Development", "Firebase", "Mobile UI/UX", "REST APIs",
    ],
    "product_management": [
        "Product Strategy", "Roadmap Planning", "User Research",
        "A/B Testing", "SQL", "Jira", "Agile", "Scrum", "OKRs",
        "Competitive Analysis", "Wireframing", "Figma",
    ],
}

ROLES = {
    "Software Engineer": ["software_engineering", "web_backend"],
    "Senior Software Engineer": ["software_engineering", "web_backend"],
    "Full Stack Developer": ["web_frontend", "web_backend"],
    "Frontend Developer": ["web_frontend", "software_engineering"],
    "Backend Developer": ["web_backend", "software_engineering"],
    "Data Scientist": ["data_science", "machine_learning"],
    "Senior Data Scientist": ["data_science", "machine_learning"],
    "Machine Learning Engineer": ["machine_learning", "data_engineering"],
    "ML Engineer": ["machine_learning", "software_engineering"],
    "Data Engineer": ["data_engineering", "software_engineering"],
    "Data Analyst": ["data_science", "product_management"],
    "DevOps Engineer": ["devops_cloud", "software_engineering"],
    "Cloud Architect": ["devops_cloud", "software_engineering"],
    "Product Manager": ["product_management", "data_science"],
    "Security Engineer": ["cybersecurity", "software_engineering"],
    "Mobile Developer": ["mobile_development", "software_engineering"],
    "iOS Developer": ["mobile_development", "web_frontend"],
    "Android Developer": ["mobile_development", "web_frontend"],
    "AI Engineer": ["machine_learning", "software_engineering"],
    "NLP Engineer": ["machine_learning", "data_science"],
}

FIRST_NAMES = [
    "Alex", "Jordan", "Priya", "Arun", "Wei", "Yuki", "Fatima", "Omar",
    "Sofia", "Liam", "Emma", "Noah", "Lucas", "Mia", "Ethan", "Olivia",
    "Aisha", "Raj", "Chen", "Diego", "Elena", "Viktor", "Kai", "Zara",
    "Marcus", "Felix", "Luna", "Ravi", "Nina", "Tariq", "Hana", "Sage",
]
LAST_NAMES = [
    "Smith", "Johnson", "Patel", "Kumar", "Chen", "Wang", "Kim", "Park",
    "Garcia", "Martinez", "Brown", "Davis", "Lee", "Wilson", "Taylor",
    "Anderson", "Müller", "Johansson", "Fernandez", "O'Brien", "Rossi",
    "Dubois", "Hansen", "Singh", "Zhang", "Tanaka", "Berg", "Costa",
]

BIO_TEMPLATES = [
    "Experienced {role} with {years} years of expertise. Proficient in {skills_phrase} and passionate about building impactful solutions.",
    "Results-driven {role} bringing {years} years of hands-on experience. Skilled in {skills_phrase} with a strong focus on delivering quality.",
    "Detail-oriented {role} with {years}+ years in the industry. Core competencies include {skills_phrase}.",
    "{role} with a {years}-year track record of delivering high-quality work. Adept at {skills_phrase} and thrives in collaborative environments.",
    "Motivated {role} seeking new challenges after {years} years of professional growth. Expert in {skills_phrase}.",
    "Innovative {role} passionate about leveraging {skills_phrase} to solve complex problems. {years} years of experience.",
    "Dedicated {role} with {years} years of experience building scalable systems. Specializing in {skills_phrase}.",
]

def generate_candidate(cid):
    role = random.choice(list(ROLES.keys()))
    domains = ROLES[role]
    pool = list(set(s for d in domains for s in SKILL_POOLS[d]))
    skills = random.sample(pool, min(random.randint(5, 8), len(pool)))
    exp = random.choices([1,2,3,4,5,6,7,8,10,12,15], weights=[5,8,12,15,15,12,10,8,6,5,4], k=1)[0]
    bio = random.choice(BIO_TEMPLATES).format(role=role, years=exp, skills_phrase=", ".join(skills[:3]))
    return {
        "id": cid,
        "name": f"{random.choice(FIRST_NAMES)} {random.choice(LAST_NAMES)}",
        "bio": bio, "skills": skills,
        "experience_years": exp, "desired_role": role,
    }

candidates = [generate_candidate(i) for i in range(500)]

with open('/kaggle/working/candidates.json', 'w') as f:
    json.dump(candidates, f, indent=2)

print(f"Generated {len(candidates)} candidates")
c = candidates[0]
print(f"\nSample: {c['name']} — {c['desired_role']} ({c['experience_years']}yr)")
print(f"  Skills: {', '.join(c['skills'])}")
print(f"  Bio: {c['bio'][:120]}...")

Generated 500 candidates

Sample: Yuki Fernandez — Frontend Developer (7yr)
  Skills: Linux, Tailwind CSS, System Design, React, Redux
  Bio: Motivated Frontend Developer seeking new challenges after 7 years of professional growth. Expert in Linux, Tailwind CSS,...


### Embedding Pipeline

In [13]:
from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd
import json



In [14]:
print("Loading sentence-transformer model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"Model loaded: all-MiniLM-L6-v2 (384-dimensional embeddings)\n")

Loading sentence-transformer model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded: all-MiniLM-L6-v2 (384-dimensional embeddings)



In [16]:
df = pd.read_csv('/kaggle/working/jobs_processed.csv')
title_col = next(c for c in ['title', 'job_title', 'Job_Ttl'] if c in df.columns)
desc_col = next(c for c in ['description', 'job_description', 'Job_Desc'] if c in df.columns)
with open('/kaggle/working/candidates.json') as f:
    candidates = json.load(f)


In [18]:
job_texts = (df[title_col].fillna('') + '. ' + df[desc_col].fillna('')).tolist()
print(f"Encoding {len(job_texts)} jobs...")
job_embeddings = model.encode(
    job_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True  
)
print(f" Job embeddings shape: {job_embeddings.shape}\n")

Encoding 1044 jobs...


Batches:   0%|          | 0/17 [00:00<?, ?it/s]

 Job embeddings shape: (1044, 384)



In [19]:
candidate_texts = [
    c['bio'] + ' ' + ' '.join(c['skills']) + ' ' + c.get('desired_role', '')
    for c in candidates
]
print(f"Encoding {len(candidate_texts)} candidates...")
candidate_embeddings = model.encode(
    candidate_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)
print(f"Candidate embeddings shape: {candidate_embeddings.shape}\n")

Encoding 500 candidates...


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Candidate embeddings shape: (500, 384)



In [20]:
np.save('/kaggle/working/job_embeddings.npy', job_embeddings)
np.save('/kaggle/working/candidate_embeddings.npy', candidate_embeddings)
print("All embeddings saved!")
print(f"   job_embeddings.npy:       {job_embeddings.shape}")
print(f"   candidate_embeddings.npy: {candidate_embeddings.shape}")

All embeddings saved!
   job_embeddings.npy:       (1044, 384)
   candidate_embeddings.npy: (500, 384)


### Build FAISS Index + Test Search

In [21]:
import faiss
import numpy as np 
import pandas as pd

In [22]:
job_embeddings = np.load('/kaggle/working/job_embeddings.npy').astype('float32')
df = pd.read_csv('/kaggle/working/jobs_processed.csv')
title_col = next(c for c in ['title', 'job_title', 'Job_Ttl'] if c in df.columns)
company_col = next((c for c in ['company_name', 'company', 'Co_Nm'] if c in df.columns), None)
dimension = job_embeddings.shape[1]  
index = faiss.IndexFlatIP(dimension)  
index.add(job_embeddings)
faiss.write_index(index, '/kaggle/working/job_index.faiss')
print(f" FAISS index built!")
print(f"   Vectors: {index.ntotal}")
print(f"   Dimension: {dimension}")
print(f"   Type: IndexFlatIP (exact cosine similarity)")

 FAISS index built!
   Vectors: 1044
   Dimension: 384
   Type: IndexFlatIP (exact cosine similarity)


In [23]:
print(f"\n Sanity check — searching for 'Python machine learning engineer':")
test_emb = model.encode(["Python machine learning engineer with NLP experience"], 
                         normalize_embeddings=True).astype('float32')
D, I = index.search(test_emb, 5)
print(f"\nTop 5 results:")
for rank, (idx, score) in enumerate(zip(I[0], D[0]), 1):
    title = df.iloc[idx][title_col]
    comp = df.iloc[idx][company_col] if company_col else "N/A"
    print(f"  {rank}. [{score:.4f}] {title} @ {comp}")


 Sanity check — searching for 'Python machine learning engineer':

Top 5 results:
  1. [0.5979] AI/ML Engineer @ ConglomerateIT
  2. [0.5627] Data Engineer @ Tech Mahindra Business Process Services
  3. [0.5525] Machine Learning Engineer @ Infosys
  4. [0.5460] Machine Learning Engineer @ StreetID
  5. [0.5449] Machine Learning Engineer @ Capgemini


### BM25 Re-ranker + Test Hybrid Ranking

In [24]:
from rank_bm25 import BM25Okapi

In [25]:
df = pd.read_csv('/kaggle/working/jobs_processed.csv')
title_col = next(c for c in ['title', 'job_title', 'Job_Ttl'] if c in df.columns)
desc_col = next(c for c in ['description', 'job_description', 'Job_Desc'] if c in df.columns)
company_col = next((c for c in ['company_name', 'company', 'Co_Nm'] if c in df.columns), None)
job_texts = (df[title_col].fillna('') + '. ' + df[desc_col].fillna('')).tolist()
tokenized_jobs = [text.lower().split() for text in job_texts]
bm25 = BM25Okapi(tokenized_jobs)
print(f"BM25 index built over {len(tokenized_jobs)} jobs\n")

BM25 index built over 1044 jobs



In [26]:
def hybrid_rank(query_text, ann_indices, semantic_scores, alpha=0.6):
    
    query_tokens = query_text.lower().split()
    full_bm25 = np.array(bm25.get_scores(query_tokens), dtype=np.float64)
    
    valid = [(i, s) for i, s in zip(ann_indices, semantic_scores) if i != -1]
    if not valid:
        return []
    
    indices, sem_scores = zip(*valid)
    indices = np.array(indices)
    sem_scores = np.array(sem_scores, dtype=np.float64)
    bm25_scores = full_bm25[indices]
    
    def normalize(arr):
        mn, mx = arr.min(), arr.max()
        return (arr - mn) / (mx - mn) if mx > mn else np.ones_like(arr)
    
    sem_norm = normalize(sem_scores)
    bm25_norm = normalize(bm25_scores)
    hybrid = alpha * sem_norm + (1 - alpha) * bm25_norm
    
    results = [(int(idx), float(score)) for idx, score in zip(indices, hybrid)]
    return sorted(results, key=lambda x: -x[1])


In [27]:
test_queries = [
    "Python machine learning engineer with deep learning and NLP experience",
    "Senior React developer with TypeScript and Node.js full stack",
    "Data engineer with Apache Spark SQL and ETL pipeline experience",
    "DevOps engineer with AWS Kubernetes Docker and Terraform",
    "Cybersecurity analyst with penetration testing and SIEM experience",
]
print(" Testing hybrid re-ranking:\n")
for query in test_queries:
    query_emb = model.encode([query], normalize_embeddings=True).astype('float32')
    D, I = index.search(query_emb, 20)
    
    # Pure semantic top-1
    sem_title = df.iloc[I[0][0]][title_col]
    
    # Hybrid top-1
    ranked = hybrid_rank(query, I[0], D[0], alpha=0.6)
    hyb_idx, hyb_score = ranked[0]
    hyb_title = df.iloc[hyb_idx][title_col]
    
    print(f"  Query:    \"{query[:60]}...\"")
    print(f"  Semantic: {sem_title}")
    print(f"  Hybrid:   {hyb_title} (score: {hyb_score:.4f})")
    print()
print("Hybrid re-ranking working!")

 Testing hybrid re-ranking:

  Query:    "Python machine learning engineer with deep learning and NLP ..."
  Semantic: AI/ML Engineer
  Hybrid:   Machine Learning Engineer (score: 0.9001)

  Query:    "Senior React developer with TypeScript and Node.js full stac..."
  Semantic: React Developer
  Hybrid:   React Developer (score: 0.9292)

  Query:    "Data engineer with Apache Spark SQL and ETL pipeline experie..."
  Semantic: Data Engineer(AWS)
  Hybrid:   Data Engineer(AWS) (score: 1.0000)

  Query:    "DevOps engineer with AWS Kubernetes Docker and Terraform..."
  Semantic: UsefulBI Corporation - Senior DevOps Engineer - AWS Cloud Services
  Hybrid:   UsefulBI Corporation - Senior DevOps Engineer - AWS Cloud Services (score: 1.0000)

  Query:    "Cybersecurity analyst with penetration testing and SIEM expe..."
  Semantic: Cyber Security Analyst
  Hybrid:   Local Defender - Cybersecurity (score: 0.8117)

Hybrid re-ranking working!


### Cold-Start Handling + Test

In [28]:
import numpy as np
job_embeddings = np.load('/kaggle/working/job_embeddings.npy')

In [29]:
def get_query_vector(profile_text, history_job_ids=None):
    """
    Build query vector from user profile + optional history.
    - No history: pure profile embedding (cold-start)
    - With history: 50% profile + 50% mean of past job embeddings (warm user)
    """
    profile_emb = model.encode([profile_text], normalize_embeddings=True)[0]
    
    if not history_job_ids:
        return profile_emb.astype(np.float32)
    
    history_embs = np.array([job_embeddings[i] for i in history_job_ids])
    blended = 0.5 * profile_emb + 0.5 * history_embs.mean(axis=0)
    norm = np.linalg.norm(blended)
    if norm > 0:
        blended = blended / norm
    return blended.astype(np.float32)
def popularity_fallback(df, top_k=5):
    """Return most popular jobs when profile is too sparse."""
    if 'applies' in df.columns:
        return df.nlargest(top_k, 'applies').index.tolist()
    if 'views' in df.columns:
        return df.nlargest(top_k, 'views').index.tolist()
    return list(range(min(top_k, len(df))))

In [33]:
print(" Testing Cold-Start Strategies:\n")
# Strategy 1: Profile-only (new user, no history)
print("Type 1: Profile-only (new user) ")
vec = get_query_vector("Senior React developer with TypeScript and Node.js")
D, I = index.search(vec.reshape(1, -1), 5)
ranked = hybrid_rank("Senior React developer with TypeScript and Node.js", I[0], D[0])
for idx, score in ranked[:5]:
    comp = df.iloc[idx][company_col] if company_col else "N/A"
    print(f"  [{score:.4f}] {df.iloc[idx][title_col]} @ {comp}")
# Strategy 2: History-augmented (returning user)
print(f"\n Type 2: History-augmented (returning user)")
vec2 = get_query_vector("Senior React developer", history_job_ids=[0, 5, 10, 15, 20])
D, I = index.search(vec2.reshape(1, -1), 5)
ranked2 = hybrid_rank("Senior React developer", I[0], D[0])
for idx, score in ranked2[:5]:
    comp = df.iloc[idx][company_col] if company_col else "N/A"
    print(f"  [{score:.4f}] {df.iloc[idx][title_col]} @ {comp}")
# Strategy 3: Popularity fallback (sparse profile)
print(f"\n Type 3: Popularity fallback (sparse profile) ")
pop_ids = popularity_fallback(df, top_k=5)
for idx in pop_ids:
    comp = df.iloc[idx][company_col] if company_col else "N/A"
    print(f"  {df.iloc[idx][title_col]} @ {comp}")
print("\nAll 3 cold-start strategies working!")


 Testing Cold-Start Strategies:

Type 1: Profile-only (new user) 
  [1.0000] React Developer @ Spot Freight
  [0.1777] Junior Frontend Developer @ Coca-Cola HBC
  [0.1185] Senior Software Engineer (AI Integration) @ Fivecast
  [0.0361] Senior Data Engineer @ Acorns
  [0.0126] **** ******** * @ ******** *******

 Type 2: History-augmented (returning user)
  [1.0000] React Developer @ Spot Freight
  [0.1726] Data Scientist, Product Analytics @ Handshake
  [0.1324] Data Scientist, Product Analytics @ Handshake
  [0.1019] Data Scientist, Product Analytics @ Handshake
  [0.0178] Senior Data Analyst (Marketing) @ foodpanda

 Type 3: Popularity fallback (sparse profile) 
  Data Analyst @ Meta
  Data Analyst @ Meta
  Data Analyst @ Meta
  Data Analyst @ Meta
  Data Analyst II @ Pinterest

All 3 cold-start strategies working!


### Evaluation

In [34]:
import json
import numpy as np
from sklearn.metrics import ndcg_score

df = pd.read_csv('/kaggle/working/jobs_processed.csv')
desc_col = next(c for c in ['description', 'job_description', 'Job_Desc'] if c in df.columns)
with open('/kaggle/working/candidates.json') as f:
    all_candidates = json.load(f)
descriptions_lower = df[desc_col].fillna('').str.lower().tolist()

In [35]:
def create_ground_truth(candidates, top_n=20):
    """A job is 'relevant' if candidate shares >=2 skills with the description."""
    gt = {}
    for c in candidates:
        skills = [s.lower() for s in c.get('skills', [])]
        matches = []
        for idx, desc in enumerate(descriptions_lower):
            count = sum(1 for s in skills if s in desc)
            if count >= 2:
                matches.append((idx, count))
        matches.sort(key=lambda x: -x[1])
        gt[c['id']] = {i for i, _ in matches[:top_n]}
    return gt
def precision_at_k(retrieved, relevant, k):
    return sum(1 for r in retrieved[:k] if r in relevant) / k if k > 0 else 0.0

In [36]:
split = int(len(all_candidates) * 0.8)
test_candidates = all_candidates[split:]
print(f"Total candidates:  {len(all_candidates)}")
print(f"Test candidates:   {len(test_candidates)}")
print("\nBuilding ground truth (skill-overlap heuristic)...")
gt = create_ground_truth(test_candidates)
gt_with_rel = {k: v for k, v in gt.items() if v}
print(f"Candidates with ≥1 relevant job: {len(gt_with_rel)}/{len(test_candidates)}")
print(f"\nRunning evaluation (k=5)...\n")
ndcg_list = []
prec_list = []
total = len(test_candidates)
for i, c in enumerate(test_candidates):
    relevant = gt.get(c['id'], set())
    if not relevant:
        continue
    
    query_text = c['bio'] + ' ' + ' '.join(c['skills']) + ' ' + c.get('desired_role', '')
    query_vec = get_query_vector(query_text)
    
    D, I = index.search(query_vec.reshape(1, -1), 20)
    ranked = hybrid_rank(query_text, I[0], D[0], alpha=0.6)
    retrieved = [idx for idx, _ in ranked[:5]]
    
    # Precision@5
    prec = precision_at_k(retrieved, relevant, 5)
    prec_list.append(prec)
    
    # NDCG@5
    if retrieved:
        true_rel = np.array([[1.0 if idx in relevant else 0.0 for idx in retrieved]])
        pred_scores = np.array([[float(5 - j) for j in range(len(retrieved))]])
        try:
            ndcg = float(ndcg_score(true_rel, pred_scores, k=5))
        except:
            ndcg = 0.0
        ndcg_list.append(ndcg)
    
    if (i + 1) % 20 == 0:
        print(f"  Progress: {i+1}/{total} candidates...")
mean_ndcg = np.mean(ndcg_list) if ndcg_list else 0.0
mean_prec = np.mean(prec_list) if prec_list else 0.0
print(f"\n{'='*55}")
print(f"   EVALUATION RESULTS (k=5)")
print(f"{'='*55}")
print(f"  Candidates evaluated : {len(prec_list)}")
print(f"  Mean NDCG@5          : {mean_ndcg:.4f}")
print(f"  Mean Precision@5     : {mean_prec:.4f}")
print(f"{'='*55}")

metrics = {"ndcg_5": round(mean_ndcg, 4), "precision_5": round(mean_prec, 4), 
           "candidates_evaluated": len(prec_list)}
with open('/kaggle/working/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print(f"\n Metrics saved to metrics.json")

Total candidates:  500
Test candidates:   100

Building ground truth (skill-overlap heuristic)...
Candidates with ≥1 relevant job: 96/100

Running evaluation (k=5)...

  Progress: 20/100 candidates...
  Progress: 40/100 candidates...
  Progress: 60/100 candidates...
  Progress: 80/100 candidates...

   EVALUATION RESULTS (k=5)
  Candidates evaluated : 96
  Mean NDCG@5          : 0.5196
  Mean Precision@5     : 0.2292

 Metrics saved to metrics.json
